# ML team adaptation guide

This notebook is the “how do we actually use this ourselves?” version.

You should not port the toy numbers.
You should port the **interfaces** and the **benchmark discipline**.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
elif not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

results_dir = project_root / "results"
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

## Step 1 — define the task honestly

Pick a task `τ` and horizon `Δ`.

Examples:
- reply in 7 days
- meeting booked in 21 days
- renewal within 90 days
- escalation within 24 hours
- candidate process advance in 14 days

If the target is sloppy, the state will get sloppy with it.

## Step 2 — define the person-side blocks

Map your domain into:
- durable profile/context blocks (`p_i, b_i, ℓ_i, r_i, h_i, g_i^grp`)
- categorical families
- source channels
- regimes / roles

The project’s toy generator uses these as stand-ins, but the real point is the separation of concern.

In [2]:
from gids_observer_framework.toy_data import generate_benchmark_dataset, default_feature_sets

toy_df = generate_benchmark_dataset(n_people=12, events_per_person=4, seed=2026)
feature_sets = default_feature_sets(toy_df)

pd.DataFrame({
    "feature_view": list(feature_sets.keys()),
    "num_columns": [len(columns) for columns in feature_sets.values()],
})

,feature_view,num_columns
0,current_touch,6
1,static,20
2,shallow_history,24
3,two_tower_like,18
4,monolithic_sequence,54
5,latent_full,12
6,latent_no_fast,12
7,latent_no_slow,10
8,latent_uniform_pool,12
9,latent_collapsed_slow,12


## Step 3 — define the proposition

A proposition can be:
- a message
- a pricing frame
- a sequence step
- a support reply template
- an offer
- a policy intervention

Keep it explicit.  
The proposition is not the same thing as the observer-state.

## Step 4 — run the weaker baselines first

Do not jump straight to a huge black box.

Start with:
- current-touch
- static tabular
- shallow history
- two-tower-like
- monolithic sequence baseline

Then add the explicit slow/fast model.

If the explicit decomposition does not win or tie with an interpretability advantage, it has not earned its place.

## Step 5 — log propensities if policy claims matter

This is the line teams usually try to skip.

If you want to say “the model chose better propositions,” then you need:
- logged propensities from the behavior policy, or
- randomized exploration / online experiments

Otherwise, call it ranking. Not causal control.

## Step 6 — use ablations as governance, not ornament

The ablations are not decorative appendices.
They are the only thing preventing a nice story from becoming theology.

Keep at least these:
- remove fast state
- remove slow state
- uniform categorical pooling
- collapsed source/regime pooling
- shuffled recent history
- monolithic sequence replacement

## Domain translation examples

In [3]:
domain_map = pd.DataFrame([
    {"domain": "GTM / sales", "observer": "buyer, founder, champion", "proposition": "message / offer / timing", "outcome": "reply / meeting / stage advance"},
    {"domain": "Support", "observer": "user or admin", "proposition": "response template / escalation", "outcome": "resolution / escalation / CSAT"},
    {"domain": "Retention", "observer": "user or account", "proposition": "nudge / education / pricing", "outcome": "activation / churn / renewal"},
    {"domain": "Recruiting", "observer": "candidate in role", "proposition": "outreach / comp framing / timing", "outcome": "reply / advance / acceptance"},
    {"domain": "Care outreach", "observer": "patient/member in context", "proposition": "reminder / channel / intervention", "outcome": "response / adherence / appointment"},
])
domain_map

,domain,observer,proposition,outcome
0,GTM / sales,"buyer, founder, champion",message / offer / timing,reply / meeting / stage advance
1,Support,user or admin,response template / escalation,resolution / escalation / CSAT
2,Retention,user or account,nudge / education / pricing,activation / churn / renewal
3,Recruiting,candidate in role,outreach / comp framing / timing,reply / advance / acceptance
4,Care outreach,patient/member in context,reminder / channel / intervention,response / adherence / appointment


## Final rule

Keep the voice of the paper where it helps:
this is a model of **predictive observer-state transition**, not a naive input/output trick.

But keep the engineering discipline even harder:
every design claim should either improve held-out results or get cut.